<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/ProcessModel_Five_Area_PFD_PID_DEXPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Five-area NeqSim ProcessModel: BFD, PFD, P&ID and DEXPI

One facility, four representations:

**A100 inlet separation → A200 recompression → A300 TEG dehydration → A400 dew-point/NGL recovery → A500 export compression**

The notebook generates a plant block diagram, plant and area PFDs, a multi-sheet P&ID proposal with reciprocal off-page connectors, and shows how the same topology maps to DEXPI 2.0 **Process** (BFD/PFD) and **Plant** (P&ID) models.

> Generated P&IDs are engineering proposals for review, not approved construction drawings.

In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
from IPython.display import SVG, display

if "google.colab" in sys.modules:
    subprocess.run([sys.executable,"-m","pip","install","-q","neqsim>=3.20.0"],check=True)
    subprocess.run(["apt-get","-qq","update"],check=False)
    subprocess.run(["apt-get","-qq","install","-y","graphviz"],check=False)

def show_dot(name, dot):
    Path(name+".dot").write_text(dot)
    subprocess.run(["dot","-Tsvg",name+".dot","-o",name+".svg"],check=True)
    display(SVG(name+".svg"))

NEQSIM_AVAILABLE = importlib.util.find_spec("neqsim") is not None
print("NeqSim available:", NEQSIM_AVAILABLE)

## 1. Block diagram

In [ ]:
show_dot("facility_block", r'''
digraph G {graph[rankdir=LR,label="Five-area ProcessModel — block diagram",labelloc=t,fontsize=20];
node[shape=box,style=rounded]; feed[label="Wet wellstream"];a100[label="A100\nInlet separation"];
a200[label="A200\nRecompression"];a300[label="A300\nTEG dehydration"];a400[label="A400\nDew-point / NGL"];
a500[label="A500\nExport compression"];sales[label="Sales gas"];liq[label="Condensate / NGL"];water[label="Produced water"];
feed->a100;a100->a200[label="HP gas"];a200->a300[label="wet compressed gas"];a300->a400[label="dry gas"];
a400->a500[label="residue gas"];a500->sales;a100->liq;a100->water;a400->liq[label="NGL"];}''')

## 2. Five-area NeqSim model

The model below uses live streams between `ProcessSystem` areas. The TEG area uses `SimpleTEGAbsorber`; the low-temperature section is intentionally compact so the notebook stays readable.

In [ ]:
if NEQSIM_AVAILABLE:
    from neqsim import jneqsim
    S=jneqsim.thermo.system.SystemSrkCPAstatoil
    Stream=jneqsim.process.equipment.stream.Stream
    Heater=jneqsim.process.equipment.heatexchanger.Heater
    Cooler=jneqsim.process.equipment.heatexchanger.Cooler
    Sep=jneqsim.process.equipment.separator.Separator
    Sep3=jneqsim.process.equipment.separator.ThreePhaseSeparator
    Compressor=jneqsim.process.equipment.compressor.Compressor
    Expander=jneqsim.process.equipment.expander.Expander
    TEG=jneqsim.process.equipment.absorber.SimpleTEGAbsorber
    PS=jneqsim.process.processmodel.ProcessSystem
    PM=jneqsim.process.processmodel.ProcessModel

    fluid=S()
    for n,z in [("nitrogen",.01),("CO2",.02),("methane",.79),("ethane",.08),
                ("propane",.04),("n-butane",.02),("n-pentane",.015),("n-hexane",.01),("water",.015)]:
        fluid.addComponent(n,z)
    fluid.setMixingRule(10)
    feed=Stream("Feed",fluid); feed.setFlowRate(100000,"kg/hr"); feed.setPressure(100,"bara"); feed.setTemperature(45,"C")

    e101=Heater("E-101 Feed heater",feed); e101.setOutTemperature(55,"C")
    v101=Sep3("V-101 HP separator",e101.getOutletStream())
    a100=PS("A100 Inlet separation")
    for u in [feed,e101,v101]: a100.add(u)

    k201=Compressor("K-201 Recompressor",v101.getGasOutStream()); k201.setOutletPressure(115)
    e202=Cooler("E-202 Aftercooler",k201.getOutletStream()); e202.setOutTemperature(35,"C")
    v202=Sep("V-202 KO drum",e202.getOutletStream())
    a200=PS("A200 Recompression")
    for u in [k201,e202,v202]: a200.add(u)

    teg=S(); teg.addComponent("water",.01); teg.addComponent("TEG",.99); teg.setMixingRule(10)
    lean=Stream("Lean TEG",teg); lean.setFlowRate(5000,"kg/hr"); lean.setPressure(113,"bara"); lean.setTemperature(35,"C")
    v301=Sep("V-301 Inlet scrubber",v202.getGasOutStream())
    c301=TEG("C-301 TEG contactor"); c301.addGasInStream(v301.getGasOutStream()); c301.addSolventInStream(lean); c301.setNumberOfStages(4)
    a300=PS("A300 TEG dehydration")
    for u in [v301,lean,c301]: a300.add(u)

    e401=Cooler("E-401 Feed cooler",c301.getGasOutStream()); e401.setOutTemperature(-10,"C")
    x401=Expander("X-401 Turboexpander",e401.getOutletStream()); x401.setOutletPressure(75)
    v401=Sep("V-401 Cold separator",x401.getOutletStream())
    a400=PS("A400 Dew-point NGL recovery")
    for u in [e401,x401,v401]: a400.add(u)

    k501=Compressor("K-501 Export compressor",v401.getGasOutStream()); k501.setOutletPressure(150)
    e502=Cooler("E-502 Export cooler",k501.getOutletStream()); e502.setOutTemperature(40,"C")
    a500=PS("A500 Export compression")
    for u in [k501,e502]: a500.add(u)

    plant=PM()
    for n,a in [("A100",a100),("A200",a200),("A300",a300),("A400",a400),("A500",a500)]: plant.add(n,a)
    plant.run()
    print("Five-area ProcessModel executed")
else:
    plant=None
    print("Run in Colab for the NeqSim calculation; all diagram cells run independently.")

## 3. Plant-wide PFD and native NeqSim export

In [ ]:
if NEQSIM_AVAILABLE:
    Path("neqsim_native_plant.dot").write_text(str(plant.toDOT()))
    subprocess.run(["dot","-Tsvg","neqsim_native_plant.dot","-o","neqsim_native_plant.svg"],check=True)
    display(SVG("neqsim_native_plant.svg"))
else:
    show_dot("plant_pfd", r'''
digraph G {graph[rankdir=LR,compound=true,label="Plant-wide PFD",labelloc=t,fontsize=20];node[fontsize=9];
subgraph cluster_100{label="A100 Separation";f[label="Feed",shape=ellipse];e101[label="E-101",shape=circle];v101[label="V-101",shape=cylinder];f->e101->v101;}
subgraph cluster_200{label="A200 Recompression";k201[label="K-201",shape=trapezium];e202[label="E-202",shape=circle];v202[label="V-202",shape=cylinder];k201->e202->v202;}
subgraph cluster_300{label="A300 TEG";v301[label="V-301",shape=cylinder];c301[label="C-301 TEG contactor",shape=box];v301->c301;}
subgraph cluster_400{label="A400 Dew-point/NGL";e401[label="E-401",shape=circle];x401[label="X-401",shape=invtrapezium];v401[label="V-401",shape=cylinder];e401->x401->v401;}
subgraph cluster_500{label="A500 Export";k501[label="K-501",shape=trapezium];e502[label="E-502",shape=circle];sales[label="Sales gas",shape=ellipse];k501->e502->sales;}
v101->k201[label="HC-2001"];v202->v301[label="HC-3001"];c301->e401[label="HC-4001"];v401->k501[label="HC-5001"];}''')

# Native NeqSim also supports plant.exportAreaDOT(Paths.get("plant-diagrams"))

## 4. Multi-sheet P&ID proposal

Each process area becomes one drawing sheet. Cross-area process streams use reciprocal off-page connector identities:
`HC-2001`, `HC-3001`, `HC-4001`, `HC-5001`.

In [ ]:
sheets={
"1":'in[label="FEED",shape=ellipse];xv[label="XV-101",shape=diamond];v[label="V-101",shape=cylinder];out[label="TO S2 HC-2001",shape=hexagon];in->xv->v->out;lt[label="LT-101",shape=circle];lic[label="LIC-101",shape=circle,style=dashed];v->lt[style=dotted];lt->lic[style=dashed];lic->xv[style=dashed];',
"2":'in[label="FROM S1 HC-2001",shape=hexagon];k[label="K-201",shape=trapezium];e[label="E-202",shape=circle];v[label="V-202",shape=cylinder];out[label="TO S3 HC-3001",shape=hexagon];in->k->e->v->out;pic[label="PIC-201",shape=circle,style=dashed];pic->k[style=dashed];',
"3":'in[label="FROM S2 HC-3001",shape=hexagon];v[label="V-301",shape=cylinder];c[label="C-301 TEG contactor",shape=box];out[label="TO S4 HC-4001",shape=hexagon];teg[label="LEAN TEG",shape=ellipse];in->v->c->out;teg->c;lic[label="LIC-302",shape=circle,style=dashed];lic->c[style=dashed];',
"4":'in[label="FROM S3 HC-4001",shape=hexagon];e[label="E-401",shape=circle];x[label="X-401",shape=invtrapezium];v[label="V-401",shape=cylinder];out[label="TO S5 HC-5001",shape=hexagon];ngl[label="NGL",shape=ellipse];in->e->x->v->out;v->ngl;tic[label="TIC-401",shape=circle,style=dashed];tic->x[style=dashed];',
"5":'in[label="FROM S4 HC-5001",shape=hexagon];v[label="V-501",shape=cylinder];k[label="K-501",shape=trapezium];e[label="E-502",shape=circle];out[label="SALES GAS",shape=ellipse];in->v->k->e->out;pic[label="PIC-501",shape=circle,style=dashed];pic->k[style=dashed];'}
for n,body in sheets.items():
    show_dot("pid_sheet_"+n, f'digraph G {{graph[rankdir=LR,label="P&ID Sheet {n}",labelloc=t,fontsize=18];node[fontsize=9];{body}}}')

## 5. DEXPI: use the same engineering identities for exchange

DEXPI is a semantic exchange model, not just a graphic.

- **DEXPI 2.0 Process profile**: BFD/PFD process steps, material ports, streams and physical-state quantities.
- **DEXPI 2.0 Plant profile**: P&ID plant items, piping, nozzles, valves, instruments, safeguards and diagram representations.

NeqSim currently provides `Dexpi20ProcessModelWriter` for the Process model and `Dexpi20XmlWriter` for the Plant/P&ID model.

In [ ]:
show_dot("dexpi_mapping", r'''
digraph G {graph[rankdir=LR,label="NeqSim → DEXPI 2.0",labelloc=t,fontsize=20];node[fontsize=10];
neq[label="NeqSim ProcessModel\nA100…A500",shape=folder];
process[label="DEXPI Process\nBFD/PFD",shape=folder];plant[label="DEXPI Plant\nP&ID",shape=folder];
steps[label="ProcessSteps + MaterialPorts + Streams",shape=box];
items[label="PlantItems + Piping + Valves",shape=box];
inst[label="Instrumentation + Safeguards",shape=box];
repr[label="Diagram representations + sheets + OPCs",shape=box];
neq->process;neq->plant;process->steps;plant->items;plant->inst;plant->repr;}''')

In [ ]:
if NEQSIM_AVAILABLE:
    from jpype import JClass
    DexpiProcessWriter=JClass("neqsim.process.processmodel.dexpi.Dexpi20ProcessModelWriter")
    DexpiPlantWriter=JClass("neqsim.process.processmodel.dexpi.Dexpi20XmlWriter")
    JavaFile=JClass("java.io.File")

    # PFD/BFD exchange: one DEXPI Process file per area is a simple, readable starting point.
    for name,area in [("A100",a100),("A200",a200),("A300",a300),("A400",a400),("A500",a500)]:
        report=DexpiProcessWriter.writeAndAssess(area,JavaFile(f"{name}_process.dexpi.xml"))
        print(name,"DEXPI Process export assessed:",report)

    # P&ID-level DEXPI Plant export normally follows engineering enrichment
    # (tags, piping/nozzles, instrumentation, safeguards, reviewed sheet layout).
    print("Dexpi20XmlWriter is available for the DEXPI Plant/P&ID profile.")
else:
    print("Run in Colab to create and assess DEXPI XML files from the live NeqSim areas.")

## 6. Recommended architecture

```text
NeqSim ProcessModel
 ├─ Block diagram
 ├─ Canonical topology / EngineeringGraph
 │   ├─ plant PFD
 │   ├─ area PFDs
 │   └─ DEXPI 2.0 Process export
 └─ engineering enrichment
     ├─ tags + line IDs + nozzles
     ├─ instruments + control/safeguarding proposals
     ├─ sheet layout + reciprocal off-page connectors
     ├─ native SVG/PDF P&ID proposal
     └─ DEXPI 2.0 Plant export
```

This keeps simulation, drawing generation and machine-readable engineering exchange linked by stable equipment/stream identities—useful for agentic facility workflows.